# APC-MuCoS: FB15k-237 Relation Prediction — Kaggle GPU version

This version is edited for Kaggle Notebooks and supports either:

1. a full project zip/dataset containing `config.py`, `train.py`, `model.py`, `data_loader.py`, `utils.py`, and optionally `data/FB15k-237/`, **or**
2. extracted project code files plus raw FB15k-237 files: `train.txt`, `valid.txt`, and `test.txt`.

**Important:** `train.txt`, `valid.txt`, and `test.txt` are only the dataset. They do not replace the project code files. If you cannot upload the original zip, upload the extracted project files as a Kaggle Dataset instead.

**Kaggle setup before running:**
1. Attach a Kaggle Dataset containing the project code files, or the original project zip.
2. Attach a Kaggle Dataset containing `train.txt`, `valid.txt`, and `test.txt` if the project dataset does not already include `data/FB15k-237/`.
3. If you have Colab checkpoints, upload them as another Kaggle Dataset too: `checkpoint_epoch_*.pth` or `checkpoint_best.pth`.
4. Notebook settings → Accelerator → **GPU P100**.
5. Notebook settings → Internet → **On**.

This notebook skips the 1-hour smoke test by default and uses AMP + larger batch size to reduce runtime.


## Cell 1 — Check GPU / Kaggle runtime


In [6]:
import os, platform, torch

print("Running on Kaggle:", os.path.exists("/kaggle/input"))
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} | {props.total_memory/1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    raise RuntimeError("No GPU detected. In Kaggle, set Accelerator = GPU before running.")


Running on Kaggle: True
Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4 | 15.6 GB
GPU 1: Tesla T4 | 15.6 GB


## Cell 2 — Install/check dependencies


In [7]:
import sys, subprocess, importlib.metadata as md

def version_tuple(v):
    return tuple(int(x) for x in v.split("+")[0].split(".")[:3] if x.isdigit())

# Kaggle usually has most packages already. Install only if transformers is missing/too old.
try:
    transformers_version = md.version("transformers")
    print("Existing transformers:", transformers_version)
    needs_install = version_tuple(transformers_version) < (4, 40, 0)
except md.PackageNotFoundError:
    transformers_version = None
    needs_install = True

if needs_install:
    print("Installing transformers/accelerate...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.40.0", "accelerate"
    ])

import transformers
print("transformers:", transformers.__version__)


Existing transformers: 5.0.0
transformers: 5.0.0


## Cell 3 — Locate Kaggle input project and raw data files


In [8]:
from pathlib import Path
import os, shutil, zipfile

KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
PROJECT_DIR = WORK_ROOT / "project"

REQUIRED_CODE_FILES = ["config.py", "data_loader.py", "model.py", "train.py", "utils.py"]
REQUIRED_DATA_FILES = ["train.txt", "valid.txt", "test.txt"]

if not KAGGLE_INPUT.exists():
    raise RuntimeError("This Kaggle version expects /kaggle/input. Attach your project/data as Kaggle Dataset(s).")

print("Attached Kaggle datasets:")
for p in sorted(KAGGLE_INPUT.iterdir()):
    print(" -", p)

# -----------------------------
# 1) Locate project code
# -----------------------------
def looks_like_project_dir(p: Path) -> bool:
    return all((p / f).exists() for f in REQUIRED_CODE_FILES)

project_candidates = []
for cfg in KAGGLE_INPUT.rglob("config.py"):
    parent = cfg.parent
    if looks_like_project_dir(parent):
        project_candidates.append(parent)

zip_candidates = sorted(
    list(KAGGLE_INPUT.rglob("*apc*mucos*fb15k237*.zip")) +
    list(KAGGLE_INPUT.rglob("*APC*MuCoS*FB15k237*.zip")) +
    list(KAGGLE_INPUT.rglob("*.zip"))
)

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if project_candidates:
    src_project = project_candidates[0]
    print("Found extracted project code:", src_project)
    shutil.copytree(src_project, PROJECT_DIR)
elif zip_candidates:
    zip_path = zip_candidates[0]
    print("Extracting project zip:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(WORK_ROOT)
    found = []
    for cfg in WORK_ROOT.rglob("config.py"):
        parent = cfg.parent
        if looks_like_project_dir(parent):
            found.append(parent)
    if not found:
        raise FileNotFoundError(
            "Zip was extracted, but project code was not found. Expected: " + ", ".join(REQUIRED_CODE_FILES)
        )
    # Move/copy the found project into /kaggle/working/project
    if found[0] != PROJECT_DIR:
        shutil.move(str(found[0]), str(PROJECT_DIR))
else:
    raise FileNotFoundError(
        "I found train/valid/test data maybe, but not the project code. "
        "You still need config.py, data_loader.py, model.py, train.py, and utils.py. "
        "Upload the original project zip OR upload the extracted project files as a Kaggle Dataset."
    )

# -----------------------------
# 2) Ensure raw dataset files exist under project/data/FB15k-237
# -----------------------------
data_dir = PROJECT_DIR / "data" / "FB15k-237"
data_dir.mkdir(parents=True, exist_ok=True)

# If raw files were uploaded as a separate Kaggle dataset, copy them into the project data folder.
copied_data = []
for name in REQUIRED_DATA_FILES:
    dst = data_dir / name
    if dst.exists():
        continue
    matches = sorted([p for p in KAGGLE_INPUT.rglob(name) if p.is_file()])
    if matches:
        shutil.copy2(matches[0], dst)
        copied_data.append(f"{name} <- {matches[0]}")

missing_data = [name for name in REQUIRED_DATA_FILES if not (data_dir / name).exists()]
if missing_data:
    raise FileNotFoundError(
        "Missing dataset files under data/FB15k-237: " + ", ".join(missing_data) +
        ". Attach train.txt, valid.txt, and test.txt as a Kaggle Dataset, or include them in the project files."
    )

os.chdir(PROJECT_DIR)
print("\nWorking directory:", Path.cwd())
print("Project files:", sorted([p.name for p in PROJECT_DIR.iterdir()])[:30])
print("Dataset folder:", data_dir)
print("Copied raw data files:", copied_data if copied_data else "none needed")
print("Dataset files:")
for name in REQUIRED_DATA_FILES:
    p = data_dir / name
    print(f" - {name}: {p.stat().st_size/1e6:.1f} MB")



Attached Kaggle datasets:
 - /kaggle/input/datasets
Found extracted project code: /kaggle/input/datasets/furqannasir1/apncas

Working directory: /kaggle/working/project
Project files: ['checkpoint_epoch_13.pth', 'config.py', 'data', 'data_loader.py', 'model.py', 'test.txt', 'train.py', 'train.txt', 'utils.py', 'valid.txt']
Dataset folder: /kaggle/working/project/data/FB15k-237
Copied raw data files: ['train.txt <- /kaggle/input/datasets/furqannasir1/apncas/train.txt', 'valid.txt <- /kaggle/input/datasets/furqannasir1/apncas/valid.txt', 'test.txt <- /kaggle/input/datasets/furqannasir1/apncas/test.txt']
Dataset files:
 - train.txt: 21.0 MB
 - valid.txt: 1.3 MB
 - test.txt: 1.5 MB


## Cell 4 — Verify the dataset

In [9]:
import pandas as pd
from pathlib import Path

root = Path('data/FB15k-237')
for name in ['train.txt', 'valid.txt', 'test.txt']:
    df = pd.read_csv(root/name, sep='\t', header=None,
                     names=['head','relation','tail'], dtype=str)
    print(f'{name}: {len(df):,} rows, {df["relation"].nunique()} unique relations')

train = pd.read_csv(root/'train.txt', sep='\t', header=None,
                    names=['head','relation','tail'], dtype=str)
valid = pd.read_csv(root/'valid.txt', sep='\t', header=None,
                    names=['head','relation','tail'], dtype=str)
test  = pd.read_csv(root/'test.txt',  sep='\t', header=None,
                    names=['head','relation','tail'], dtype=str)

unseen_v = sorted(set(valid['relation']) - set(train['relation']))
unseen_t = sorted(set(test['relation'])  - set(train['relation']))
assert len(train)==272115 and len(valid)==17535 and len(test)==20466, 'Size mismatch'
assert not unseen_v, f'Unseen valid relations: {unseen_v}'
assert not unseen_t, f'Unseen test  relations: {unseen_t}'
print('\nDataset OK: sizes and relation coverage all correct.')

train.txt: 272,115 rows, 237 unique relations
valid.txt: 17,535 rows, 223 unique relations
test.txt: 20,466 rows, 224 unique relations

Dataset OK: sizes and relation coverage all correct.


## Cell 5 — Static checks

In [10]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'py_compile',
     'config.py','data_loader.py','model.py','train.py','utils.py'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('SYNTAX ERROR:', result.stderr)
else:
    print('py_compile: all files OK')

import config
config.print_config_summary()

py_compile: all files OK
Using Device: cuda
Number of GPUs: 2
Global Batch Size: 64
Per GPU Batch Size: 32
DDP Master Port: 29500
Using Model: distilbert-base-uncased
Context Mode: adaptive
Dynamic Padding: True; Pretokenize: False
Pair Context: True; Schema Prior: True
Found 5 main datasets → will train sequentially



## Cell 6 — Optional smoke run (skipped by default)


In [4]:
# The previous Colab smoke run cost about 56 minutes on a T4.
# To minimize time on Kaggle, keep this False unless you changed code and need a quick verification.
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    import importlib, config, os

    config.NUM_EPOCHS = 1
    config.BATCH_SIZE = 16
    config.PER_GPU_BATCH_SIZE = 16
    config.EARLY_STOPPING_PATIENCE = 1
    config.GRADIENT_ACCUMULATION_STEPS = 1
    config.NUM_WORKERS = min(2, os.cpu_count() or 2)
    if hasattr(config, "SAVE_EVERY_EPOCH"):
        config.SAVE_EVERY_EPOCH = True

    import train
    importlib.reload(train)
    train.train_single_process(config.DATASETS[0])
    print("\nSmoke run complete.")
else:
    print("Skipping smoke run to save GPU time.")


Skipping smoke run to save GPU time.


## Cell 7A — Fast full-run settings


In [11]:
import importlib, os, torch, config

# Fast settings for Kaggle GPU:
# - AMP is used in Cell 7C.
# - Batch 64 + grad accumulation 1 keeps the same effective batch as the original 32 x accum 2.
# - If you get CUDA OOM, change FAST_BATCH_SIZE to 32 and rerun from this cell.
FAST_BATCH_SIZE = 64

config.NUM_EPOCHS = 20
config.BATCH_SIZE = FAST_BATCH_SIZE
config.PER_GPU_BATCH_SIZE = FAST_BATCH_SIZE
config.GRADIENT_ACCUMULATION_STEPS = 1
config.EARLY_STOPPING_PATIENCE = 10
config.EARLY_STOPPING_MIN_EPOCH = 10
config.NUM_WORKERS = min(2, os.cpu_count() or 2)
config.SAVE_EVERY_EPOCH = True

# Long-tail / rare-relation improvements from v2, applied to efficient v1.
config.LOSS_TYPE = "deferred_weighted_ce"
config.REWEIGHT_START_EPOCH = 3
config.TRAIN_SAMPLER = "relation_balanced"
config.RELATION_SAMPLER_POWER = 0.5

# Optional speed-up if the project supports it.
if hasattr(config, "PRETOKENIZE"):
    config.PRETOKENIZE = True
if hasattr(config, "DYNAMIC_PADDING"):
    config.DYNAMIC_PADDING = True

# Keep evaluation quality the same.
if hasattr(config, "FILTERED_RELATION_EVAL"):
    config.FILTERED_RELATION_EVAL = True

print("Full-run config:")
print("  NUM_EPOCHS:", config.NUM_EPOCHS)
print("  BATCH_SIZE:", config.BATCH_SIZE)
print("  GRADIENT_ACCUMULATION_STEPS:", config.GRADIENT_ACCUMULATION_STEPS)
print("  Effective batch:", config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS)
print("  NUM_WORKERS:", config.NUM_WORKERS)
print("  SAVE_EVERY_EPOCH:", config.SAVE_EVERY_EPOCH)
print("  PRETOKENIZE:", getattr(config, "PRETOKENIZE", "not in config"))


Full-run config:
  NUM_EPOCHS: 20
  BATCH_SIZE: 64
  GRADIENT_ACCUMULATION_STEPS: 1
  Effective batch: 64
  NUM_WORKERS: 2
  SAVE_EVERY_EPOCH: True
  PRETOKENIZE: True


## Cell 7B — Import checkpoints from Kaggle input


In [12]:
from pathlib import Path
import os, shutil, glob, re, json
import config

PROJECT_DIR = Path.cwd()
LOCAL_OUT = PROJECT_DIR / "outputs" / "FB15k-237" / "apc-mucos-distilbert"
KAGGLE_SAVE_OUT = Path("/kaggle/working/APC_MuCoS_FB15k237")

LOCAL_OUT.mkdir(parents=True, exist_ok=True)
KAGGLE_SAVE_OUT.mkdir(parents=True, exist_ok=True)

# Copy checkpoints/metrics from any attached Kaggle Dataset into the model output folder.
patterns = [
    "checkpoint_epoch_*.pth",
    "checkpoint_best.pth",
    "relation_val_metrics.jsonl",
    "relation_metrics.jsonl",
    "run_config.json",
]
copied = []
for pat in patterns:
    for src in Path("/kaggle/input").rglob(pat):
        dst = LOCAL_OUT / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            copied.append(src.name)

# Also restore files from /kaggle/working/APC_MuCoS_FB15k237 if this cell is rerun in the same session.
for pat in patterns:
    for src in KAGGLE_SAVE_OUT.glob(pat):
        dst = LOCAL_OUT / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            copied.append(src.name)

print("Model output folder:", LOCAL_OUT)
print("Kaggle save folder:", KAGGLE_SAVE_OUT)
print("Copied files:", copied if copied else "none")

print("Available checkpoints:")
for p in sorted(LOCAL_OUT.glob("checkpoint*.pth")):
    print(" -", p.name, f"({p.stat().st_size/1e6:.1f} MB)")


Model output folder: /kaggle/working/project/outputs/FB15k-237/apc-mucos-distilbert
Kaggle save folder: /kaggle/working/APC_MuCoS_FB15k237
Copied files: ['checkpoint_epoch_13.pth']
Available checkpoints:
 - checkpoint_epoch_13.pth (805.9 MB)


## Cell 7C — Resume-capable Kaggle training loop


In [ ]:
import importlib, config, train, os, shutil, glob, re, json, time
from pathlib import Path
import torch

# Reload train.py so it sees the current config module, but do NOT reload config.py
# because earlier Kaggle cells may already have patched speed settings.
importlib.reload(train)

# ---------------------------------------------------------------------
# Kaggle paths + config override
# ---------------------------------------------------------------------
# Cell 3 should have set the working directory to /kaggle/working/project.
# This fallback makes the cell safe even if Kaggle starts from /kaggle/working.
PROJECT_DIR = Path.cwd()
if (Path("/kaggle/working/project") / "config.py").exists():
    PROJECT_DIR = Path("/kaggle/working/project")
os.chdir(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data" / "FB15k-237"
LOCAL_OUT = PROJECT_DIR / "outputs" / "FB15k-237" / "apc-mucos-distilbert"
KAGGLE_SAVE_OUT = Path("/kaggle/working/APC_MuCoS_FB15k237")
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
KAGGLE_SAVE_OUT.mkdir(parents=True, exist_ok=True)

# If train/valid/test are in the project root or in /kaggle/input, copy them into data/FB15k-237.
def _find_file(name):
    candidates = [PROJECT_DIR / name, DATA_DIR / name]
    try:
        candidates += list(Path("/kaggle/input").rglob(name))
    except Exception:
        pass
    for p in candidates:
        if p.exists() and p.is_file():
            return p
    return None

for fname in ["train.txt", "valid.txt", "test.txt"]:
    dst = DATA_DIR / fname
    if not dst.exists():
        src = _find_file(fname)
        if src is not None and src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
            print(f"Copied {fname} -> {dst}")

# CRITICAL FIX: override old /home/user/... paths from config.py with Kaggle paths.
dataset_config = config.DATASETS[0]
dataset_config.update({
    "name": "FB15k-237",
    "TRAIN_FILE_PATH": str(DATA_DIR / "train.txt"),
    "VALID_FILE_PATH": str(DATA_DIR / "valid.txt"),
    "TEST_FILE_PATH": str(DATA_DIR / "test.txt"),
    "MODEL_SAVE_PATH": str(LOCAL_OUT),
    "ENTITY_TYPE_FILE": None,
    "ENTITY_LABEL_FILE": None,
    "RELATION_LABEL_FILE": None,
})
config.DATASETS[0] = dataset_config

# Force single compatible CUDA device. T4 x2 is present, but this loop is single-GPU.
if torch.cuda.is_available():
    config.DEVICE = torch.device("cuda:0")
else:
    config.DEVICE = torch.device("cpu")

print("Using Kaggle paths:")
for k in ["TRAIN_FILE_PATH", "VALID_FILE_PATH", "TEST_FILE_PATH", "MODEL_SAVE_PATH"]:
    p = Path(dataset_config[k])
    print(f" - {k}: {p} | exists={p.exists()}")

missing = [k for k in ["TRAIN_FILE_PATH", "VALID_FILE_PATH", "TEST_FILE_PATH"] if not Path(dataset_config[k]).exists()]
if missing:
    raise FileNotFoundError(f"Missing dataset files after Kaggle path override: {missing}")

# Copy checkpoints from project root / Kaggle input into LOCAL_OUT before resume detection.
checkpoint_candidates = []
checkpoint_candidates += list(PROJECT_DIR.glob("checkpoint_best.pth"))
checkpoint_candidates += list(PROJECT_DIR.glob("checkpoint_epoch_*.pth"))
try:
    checkpoint_candidates += list(Path("/kaggle/input").rglob("checkpoint_best.pth"))
    checkpoint_candidates += list(Path("/kaggle/input").rglob("checkpoint_epoch_*.pth"))
except Exception:
    pass

for ckpt in checkpoint_candidates:
    dst = LOCAL_OUT / ckpt.name
    if ckpt.exists() and ckpt.resolve() != dst.resolve() and not dst.exists():
        shutil.copy2(ckpt, dst)
        print(f"Copied checkpoint into output folder: {dst.name}")

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def infer_checkpoint_epoch(path):
    """Return saved epoch if present inside a checkpoint; otherwise None."""
    try:
        ckpt = torch.load(path, map_location="cpu")
        if isinstance(ckpt, dict):
            for key in ["epoch", "current_epoch", "completed_epoch"]:
                if key in ckpt:
                    return int(ckpt[key])
    except Exception as e:
        print(f"Could not inspect checkpoint epoch for {Path(path).name}: {e}")
    return None

def copy_to_kaggle_output(path):
    path = Path(path)
    if path.exists():
        dst = KAGGLE_SAVE_OUT / path.name
        shutil.copy2(path, dst)
        return dst
    return None

def sync_key_outputs():
    for pat in [
        "checkpoint_epoch_*.pth", "checkpoint_best.pth",
        "relation_val_metrics.jsonl", "relation_metrics.jsonl",
        "relation_test_results.txt", "relation_val_test_results.txt", "run_config.json"
    ]:
        for p in LOCAL_OUT.glob(pat):
            copy_to_kaggle_output(p)

# ---------------------------------------------------------------------
# Choose best resume source
# ---------------------------------------------------------------------
epoch_ckpts = sorted(
    LOCAL_OUT.glob("checkpoint_epoch_*.pth"),
    key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)\.pth", p.name).group(1))
)

resume_ckpt = None
resume_mode = "base"
start_epoch = 0

if epoch_ckpts:
    resume_ckpt = epoch_ckpts[-1]
    start_epoch = int(re.search(r"checkpoint_epoch_(\d+)\.pth", resume_ckpt.name).group(1))
    resume_mode = "exact_epoch"
elif (LOCAL_OUT / "checkpoint_best.pth").exists():
    resume_ckpt = LOCAL_OUT / "checkpoint_best.pth"
    inferred = infer_checkpoint_epoch(resume_ckpt)
    start_epoch = inferred if inferred is not None else 0
    resume_mode = "best_checkpoint"

if resume_ckpt:
    print(f"Resume source: {resume_ckpt.name}")
    if resume_mode == "exact_epoch":
        print(f"Exact resume from completed epoch {start_epoch}; next epoch is {start_epoch + 1}.")
    else:
        if start_epoch > 0:
            print(f"Warm/exact resume from checkpoint_best.pth saved at epoch {start_epoch}; next epoch is {start_epoch + 1}.")
        else:
            print("Warm-starting from checkpoint_best.pth, but epoch number was not found; training schedule starts at epoch 1.")
else:
    print("No checkpoint found — starting from the base DistilBERT model.")

# ---------------------------------------------------------------------
# Build data/model objects
# ---------------------------------------------------------------------
from utils import load_checkpoint, save_checkpoint, set_seed
from data_loader import (
    load_triplets, build_relation_to_idx, load_entity_types, load_label_map,
    build_known_true_relations_by_pair, build_schema_prior_counts,
    validate_relation_coverage
)
from model import get_model_and_tokenizer
from collections import Counter

set_seed(config.SEED)
device = config.DEVICE
os.makedirs(dataset_config["MODEL_SAVE_PATH"], exist_ok=True)

# These now point to /kaggle/working/project/data/FB15k-237/*.txt
train_triplets = load_triplets(dataset_config["TRAIN_FILE_PATH"])
valid_triplets = load_triplets(dataset_config["VALID_FILE_PATH"])
test_triplets  = load_triplets(dataset_config["TEST_FILE_PATH"])

print(f"Loaded triples: train={len(train_triplets):,}, valid={len(valid_triplets):,}, test={len(test_triplets):,}")

relation_to_idx = build_relation_to_idx(train_triplets)
validate_relation_coverage(relation_to_idx, valid_triplets, test_triplets)
num_relations = len(relation_to_idx)
relation_train_counts = dict(Counter(train_triplets["relation"].tolist()))
known_true_relations_by_pair = build_known_true_relations_by_pair(
    train_triplets, valid_triplets, test_triplets
)

entity_types = load_entity_types(dataset_config.get("ENTITY_TYPE_FILE"))
entity_label_map   = load_label_map(dataset_config.get("ENTITY_LABEL_FILE"))
relation_label_map = load_label_map(dataset_config.get("RELATION_LABEL_FILE"))

graph_info = train.load_or_build_graph_info(0, False, train_triplets, entity_types, dataset_config)

schema_pair_relation_counts = None
if getattr(config, "USE_SCHEMA_PRIOR", True):
    schema_pair_relation_counts = build_schema_prior_counts(graph_info, relation_to_idx)

model, tokenizer = get_model_and_tokenizer(
    config.MODEL_NAME,
    num_labels=num_relations,
    special_tokens=config.SPECIAL_TOKENS
)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE)

# Load checkpoint weights/optimizer if available.
if resume_ckpt and Path(resume_ckpt).exists():
    try:
        loaded_epoch = load_checkpoint(model, optimizer, str(resume_ckpt), map_location=device)
        if loaded_epoch:
            start_epoch = max(start_epoch, loaded_epoch)
        print(f"Loaded checkpoint with optimizer: {Path(resume_ckpt).name}; loaded_epoch={loaded_epoch}")
    except Exception as e:
        print(f"Optimizer load failed ({e}); trying model-only load.")
        loaded_epoch = load_checkpoint(model, None, str(resume_ckpt), map_location=device)
        if loaded_epoch:
            start_epoch = max(start_epoch, loaded_epoch)
        print(f"Loaded checkpoint model weights only: {Path(resume_ckpt).name}; loaded_epoch={loaded_epoch}")

# Create loaders after config/path patch.
train_loader, valid_loader, test_loader, _ = train.create_datasets_and_loaders(
    0, 1, False, tokenizer, relation_to_idx, graph_info, entity_types,
    entity_label_map, relation_label_map,
    dataset_config.get("name", ""), train_triplets, valid_triplets, test_triplets
)

from utils import (
    checkpoint_selection_score, compute_loss, evaluate_relation_model,
    save_test_results, apply_schema_relation_prior, compute_relation_class_weights
)

# Class weights are required for weighted_ce / deferred_weighted_ce / focal.
# This preserves CE behavior before REWEIGHT_START_EPOCH when deferred_weighted_ce is used.
class_weights = None
if getattr(config, "LOSS_TYPE", "ce") in {"weighted_ce", "deferred_weighted_ce", "focal"}:
    class_weights = compute_relation_class_weights(
        train_triplets,
        relation_to_idx,
        smoothing=getattr(config, "CLASS_WEIGHT_SMOOTHING", 1.2),
        device=device,
    )
    print(f"Class weights computed for {len(class_weights)} relations.")

# ---------------------------------------------------------------------
# Restore best validation state
# ---------------------------------------------------------------------
best_score = float("-inf")
epochs_without_improvement = 0
best_ckpt = LOCAL_OUT / "checkpoint_best.pth"
val_jsonl = LOCAL_OUT / "relation_val_metrics.jsonl"

if val_jsonl.exists():
    val_lines = [l for l in val_jsonl.read_text().splitlines() if l.strip()]
    if val_lines:
        val_rows = [json.loads(l) for l in val_lines]
        scores = [
            checkpoint_selection_score(
                r,
                metric=config.CHECKPOINT_METRIC,
                mrr_weight=config.CHECKPOINT_MRR_WEIGHT,
                macro_f1_weight=config.CHECKPOINT_MACRO_F1_WEIGHT,
            )
            for r in val_rows
        ]
        best_score = max(scores)
        last_best_idx = max(range(len(scores)), key=lambda i: scores[i])
        epochs_without_improvement = len(scores) - 1 - last_best_idx
        print(f"Restored best validation score: {best_score:.4f}")
        print(f"Restored epochs without improvement: {epochs_without_improvement}")

if start_epoch >= config.NUM_EPOCHS:
    print(f"Checkpoint is already at epoch {start_epoch}, which is >= NUM_EPOCHS={config.NUM_EPOCHS}. Skipping training.")
else:
    print(f"\nTraining epochs {start_epoch + 1} to {config.NUM_EPOCHS}")

# ---------------------------------------------------------------------
# Fast AMP training loop
# ---------------------------------------------------------------------
use_amp = torch.cuda.is_available()
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    autocast_context = lambda: torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    autocast_context = lambda: torch.cuda.amp.autocast(enabled=use_amp)

print("Device:", device)
print("AMP enabled:", use_amp)
print("Train batches per epoch:", len(train_loader))
print("Batch size:", config.BATCH_SIZE, "| grad accumulation:", config.GRADIENT_ACCUMULATION_STEPS)

gradient_accumulation_steps = config.GRADIENT_ACCUMULATION_STEPS
MIN_EPOCH = getattr(config, "EARLY_STOPPING_MIN_EPOCH", 0)

for epoch in range(start_epoch, config.NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss = 0.0
    total_batches = len(train_loader)
    start_time = time.perf_counter()
    loss_type = train._effective_loss_type(epoch)

    for batch_idx, (inputs, labels, metas) in enumerate(train_loader):
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}
        labels = labels.to(device, non_blocking=True)

        with autocast_context():
            outputs = model(**inputs)
            logits = outputs.logits
            logits = apply_schema_relation_prior(
                logits, metas, schema_pair_relation_counts,
                weight=config.SCHEMA_PRIOR_WEIGHT if getattr(config, "USE_SCHEMA_PRIOR", True) else 0.0,
                smoothing=config.SCHEMA_PRIOR_SMOOTHING,
            )
            loss = compute_loss(
                logits,
                labels,
                loss_type=loss_type,
                class_weights=class_weights,
                label_smoothing=config.LABEL_SMOOTHING,
                focal_gamma=config.FOCAL_GAMMA,
            )
            scaled_loss = loss / gradient_accumulation_steps

        scaler.scale(scaled_loss).backward()

        do_step = ((batch_idx + 1) % gradient_accumulation_steps == 0) or ((batch_idx + 1) == total_batches)
        if do_step:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_loss += float(loss.item())

        if (batch_idx + 1) % 500 == 0 or (batch_idx + 1) == total_batches:
            elapsed = time.perf_counter() - start_time
            batches_per_sec = (batch_idx + 1) / max(1, elapsed)
            print(
                f"Epoch {epoch + 1} | Batch {batch_idx + 1}/{total_batches} | "
                f"Loss {loss.item():.4f} | Avg {train_loss/(batch_idx+1):.4f} | "
                f"{batches_per_sec:.2f} batches/s"
            )

    epoch_secs = time.perf_counter() - start_time
    avg_loss = train_loss / max(1, total_batches)
    print(f"\nEpoch {epoch + 1} done — avg loss: {avg_loss:.4f} — {epoch_secs:.0f}s")

    # Save per-epoch checkpoint locally and to Kaggle output.
    ckpt_path = LOCAL_OUT / f"checkpoint_epoch_{epoch + 1}.pth"
    save_checkpoint(model, optimizer, epoch + 1, str(ckpt_path))
    copy_to_kaggle_output(ckpt_path)
    print(f"Saved {ckpt_path.name}")

    # Validation
    print("Evaluating validation set...")
    val_results = evaluate_relation_model(
        model, valid_loader, device, relation_to_idx,
        schema_pair_relation_counts=schema_pair_relation_counts if getattr(config, "USE_SCHEMA_PRIOR", True) else None,
        schema_prior_weight=config.SCHEMA_PRIOR_WEIGHT,
        schema_prior_smoothing=config.SCHEMA_PRIOR_SMOOTHING,
        known_true_relations_by_pair=known_true_relations_by_pair,
        filtered_relation_eval=config.FILTERED_RELATION_EVAL,
        relation_train_counts=relation_train_counts,
        rare_threshold=config.RARE_RELATION_THRESHOLD,
        medium_threshold=config.MEDIUM_RELATION_THRESHOLD,
    )
    val_results["TrainLoss"] = avg_loss
    train.print_metrics(f"Epoch {epoch + 1} Validation:", val_results)
    save_test_results(epoch, val_results, str(LOCAL_OUT), task="relation_val")
    copy_to_kaggle_output(val_jsonl)

    current_score = checkpoint_selection_score(
        val_results,
        metric=config.CHECKPOINT_METRIC,
        mrr_weight=config.CHECKPOINT_MRR_WEIGHT,
        macro_f1_weight=config.CHECKPOINT_MACRO_F1_WEIGHT,
    )

    if current_score > best_score + config.EARLY_STOPPING_MIN_DELTA:
        best_score = current_score
        epochs_without_improvement = 0
        save_checkpoint(model, optimizer, epoch + 1, str(best_ckpt))
        copy_to_kaggle_output(best_ckpt)
        print(f"New best! Score={best_score:.4f}; saved {best_ckpt.name}")
    else:
        epochs_without_improvement += 1
        print(
            f"No improvement for {epochs_without_improvement}/{config.EARLY_STOPPING_PATIENCE} epochs "
            f"(early stop active from epoch {MIN_EPOCH})"
        )
        if (epoch + 1 >= MIN_EPOCH) and (epochs_without_improvement >= config.EARLY_STOPPING_PATIENCE):
            print("Early stopping triggered.")
            break

    sync_key_outputs()

# ---------------------------------------------------------------------
# Final test evaluation
# ---------------------------------------------------------------------
if best_ckpt.exists():
    print("\nLoading best checkpoint for test evaluation...")
    load_checkpoint(model, None, str(best_ckpt), map_location=device)
else:
    print("\nNo checkpoint_best.pth found; using current model for test evaluation.")

test_results = evaluate_relation_model(
    model, test_loader, device, relation_to_idx,
    schema_pair_relation_counts=schema_pair_relation_counts if getattr(config, "USE_SCHEMA_PRIOR", True) else None,
    schema_prior_weight=config.SCHEMA_PRIOR_WEIGHT,
    schema_prior_smoothing=config.SCHEMA_PRIOR_SMOOTHING,
    known_true_relations_by_pair=known_true_relations_by_pair,
    filtered_relation_eval=config.FILTERED_RELATION_EVAL,
    relation_train_counts=relation_train_counts,
    rare_threshold=config.RARE_RELATION_THRESHOLD,
    medium_threshold=config.MEDIUM_RELATION_THRESHOLD,
)

train.print_metrics("FINAL TEST RESULTS:", test_results)
test_jsonl = LOCAL_OUT / "relation_metrics.jsonl"
save_test_results(config.NUM_EPOCHS - 1, test_results, str(LOCAL_OUT), task="relation")
copy_to_kaggle_output(test_jsonl)
sync_key_outputs()

print("\nAll key outputs copied to:", KAGGLE_SAVE_OUT)
print("Full run complete.")


Using Kaggle paths:
 - TRAIN_FILE_PATH: /kaggle/working/project/data/FB15k-237/train.txt | exists=True
 - VALID_FILE_PATH: /kaggle/working/project/data/FB15k-237/valid.txt | exists=True
 - TEST_FILE_PATH: /kaggle/working/project/data/FB15k-237/test.txt | exists=True
 - MODEL_SAVE_PATH: /kaggle/working/project/outputs/FB15k-237/apc-mucos-distilbert | exists=True
Resume source: checkpoint_epoch_13.pth
Exact resume from completed epoch 13; next epoch is 14.
Loaded triples: train=272,115, valid=17,535, test=20,466
Precomputing train-only graph information...
Graph precompute complete in 7.88s. Saved to /kaggle/working/project/outputs/FB15k-237/apc-mucos-distilbert/graph_stats_train_only_89c18c5d00e3.pt


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `m

Loaded checkpoint with optimizer: checkpoint_epoch_13.pth; loaded_epoch=13
Class weights computed for 237 relations.

Training epochs 14 to 20
Device: cuda:0
AMP enabled: True
Train batches per epoch: 4252
Batch size: 64 | grad accumulation: 1
Epoch 14 | Batch 500/4252 | Loss 0.1663 | Avg 0.2213 | 4.93 batches/s
Epoch 14 | Batch 1000/4252 | Loss 0.1452 | Avg 0.2264 | 5.02 batches/s
Epoch 14 | Batch 1500/4252 | Loss 0.2475 | Avg 0.2243 | 5.05 batches/s
Epoch 14 | Batch 2000/4252 | Loss 0.1906 | Avg 0.2235 | 5.07 batches/s
Epoch 14 | Batch 2500/4252 | Loss 0.1449 | Avg 0.2238 | 5.08 batches/s
Epoch 14 | Batch 3000/4252 | Loss 0.1572 | Avg 0.2250 | 5.09 batches/s
Epoch 14 | Batch 3500/4252 | Loss 0.2803 | Avg 0.2242 | 5.10 batches/s
Epoch 14 | Batch 4000/4252 | Loss 0.1945 | Avg 0.2238 | 5.10 batches/s
Epoch 14 | Batch 4252/4252 | Loss 0.2275 | Avg 0.2233 | 5.11 batches/s

Epoch 14 done — avg loss: 0.2233 — 833s
Saved checkpoint_epoch_14.pth
Evaluating validation set...
Epoch 14 Validatio

## Cell 8 — Report paper-table results

## Cell 9 — Package Kaggle outputs


In [29]:
from pathlib import Path
import shutil, os

OUT_DIR = Path("outputs/FB15k-237/apc-mucos-distilbert")
KAGGLE_SAVE_OUT = Path("/kaggle/working/APC_MuCoS_FB15k237")
KAGGLE_SAVE_OUT.mkdir(parents=True, exist_ok=True)

# Copy latest output files into a clean Kaggle output folder.
for pat in [
    "checkpoint_epoch_*.pth", "checkpoint_best.pth",
    "relation_metrics.jsonl", "relation_val_metrics.jsonl",
    "relation_test_results.txt", "run_config.json"
]:
    for p in OUT_DIR.glob(pat):
        shutil.copy2(p, KAGGLE_SAVE_OUT / p.name)

archive_base = "/kaggle/working/APC_MuCoS_FB15k237_results"
archive_path = shutil.make_archive(archive_base, "zip", KAGGLE_SAVE_OUT)

print("Packaged results:", archive_path)
print("Files included:")
for p in sorted(KAGGLE_SAVE_OUT.iterdir()):
    print(" -", p.name, f"({p.stat().st_size/1e6:.1f} MB)")
print("\\nIn Kaggle, download the zip from the right-side Output panel after saving/running the notebook.")


Packaged results: /kaggle/working/APC_MuCoS_FB15k237_results.zip
Files included:
 - checkpoint_best.pth (805.9 MB)
 - checkpoint_epoch_2.pth (805.9 MB)
 - checkpoint_epoch_3.pth (805.9 MB)
 - checkpoint_epoch_4.pth (805.9 MB)
 - checkpoint_epoch_5.pth (805.9 MB)
 - checkpoint_epoch_6.pth (805.9 MB)
 - checkpoint_epoch_7.pth (805.9 MB)
 - relation_val_metrics.jsonl (0.0 MB)
 - relation_val_test_results.txt (0.0 MB)
\nIn Kaggle, download the zip from the right-side Output panel after saving/running the notebook.
